In [1]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import sys
import subprocess
import pandas as pd

direktori_aktif = Path.cwd()
direktori_project = direktori_aktif.parent if direktori_aktif.name.lower() == "notebooks" else direktori_aktif

direktori_src = direktori_project / "src"
direktori_intelligence = direktori_project / "data" / "intelligence"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_examples = direktori_project / "examples"

for folder in [direktori_src, direktori_intelligence, direktori_outputs, direktori_examples]:
    folder.mkdir(parents=True, exist_ok=True)

lokasi_engine_v5 = direktori_src / "phishrisk_engine_v5.py"
lokasi_cli_v5 = direktori_src / "run_phishrisk_v5.py"
lokasi_step17 = direktori_outputs / "hasil_validasi_url_engine_v5_best_step17.csv"

file_wajib = [lokasi_engine_v5, lokasi_cli_v5, lokasi_step17]
validasi_awal = pd.DataFrame([{
    "nama_file": file.name,
    "lokasi": str(file),
    "tersedia": file.exists(),
    "ukuran_kb": round(file.stat().st_size / 1024, 2) if file.exists() else 0,
} for file in file_wajib])

display(validasi_awal)

if not validasi_awal["tersedia"].all():
    raise FileNotFoundError("Ada file wajib STEP 17B yang belum tersedia.")

print("Semua file wajib tersedia.")


,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v5.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py,True,24.69
1,run_phishrisk_v5.py,C:\Users\ASUS\PHISHING\src\run_phishrisk_v5.py,True,3.27
2,hasil_validasi_url_engine_v5_best_step17.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_v...,True,58.98


Semua file wajib tersedia.


In [2]:
trusted_domains = [
    ("pydata.org", "data_science_reference", "Ekosistem PyData."),
    ("pandas.pydata.org", "data_science_reference", "Dokumentasi pandas."),
    ("python.org", "developer_reference", "Situs resmi Python."),
    ("docs.python.org", "developer_reference", "Dokumentasi Python."),
    ("scikit-learn.org", "data_science_reference", "Dokumentasi scikit-learn."),
    ("numpy.org", "data_science_reference", "Dokumentasi NumPy."),
    ("scipy.org", "data_science_reference", "Dokumentasi SciPy."),
    ("matplotlib.org", "data_science_reference", "Dokumentasi Matplotlib."),
    ("jupyter.org", "developer_reference", "Ekosistem Jupyter."),
    ("anaconda.org", "developer_reference", "Ekosistem Anaconda."),
    ("huggingface.co", "ai_ml_reference", "Platform AI/ML."),
    ("kaggle.com", "data_science_reference", "Platform dataset data science."),
    ("wikipedia.org", "knowledge_reference", "Referensi umum."),
    ("github.com", "developer_reference", "Repository developer."),
    ("github.io", "developer_pages", "GitHub Pages, tetap cek path."),
    ("readthedocs.io", "developer_docs", "Dokumentasi teknis."),
    ("tensorflow.org", "ai_ml_reference", "Dokumentasi TensorFlow."),
    ("pytorch.org", "ai_ml_reference", "Dokumentasi PyTorch."),
]

data_trusted = pd.DataFrame(trusted_domains, columns=["domain", "kategori", "catatan"])
lokasi_trusted = direktori_intelligence / "trusted_safe_domains_global.csv"
data_trusted.to_csv(lokasi_trusted, index=False, encoding="utf-8")

print("Trusted safe domains disimpan:", lokasi_trusted)
display(data_trusted)


Trusted safe domains disimpan: C:\Users\ASUS\PHISHING\data\intelligence\trusted_safe_domains_global.csv


,domain,kategori,catatan
0,pydata.org,data_science_reference,Ekosistem PyData.
1,pandas.pydata.org,data_science_reference,Dokumentasi pandas.
2,python.org,developer_reference,Situs resmi Python.
3,docs.python.org,developer_reference,Dokumentasi Python.
4,scikit-learn.org,data_science_reference,Dokumentasi scikit-learn.
5,numpy.org,data_science_reference,Dokumentasi NumPy.
6,scipy.org,data_science_reference,Dokumentasi SciPy.
7,matplotlib.org,data_science_reference,Dokumentasi Matplotlib.
8,jupyter.org,developer_reference,Ekosistem Jupyter.
9,anaconda.org,developer_reference,Ekosistem Anaconda.


In [3]:
isi = lokasi_engine_v5.read_text(encoding="utf-8")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = direktori_src / f"phishrisk_engine_v5_backup_before_fp_patch_{timestamp}.py"
shutil.copy2(lokasi_engine_v5, backup)

def wajib_replace(teks, lama, baru, nama):
    if lama not in teks:
        raise RuntimeError(f"Marker tidak ditemukan: {nama}")
    return teks.replace(lama, baru)

if "trusted_safe_domains" not in isi:
    isi = wajib_replace(
        isi,
        "        self.suspicious_weights = {}\n",
        "        self.suspicious_weights = {}\n        self.trusted_safe_domains = []\n",
        "atribut trusted safe",
    )

    load_trusted = """
        lokasi_trusted = self.direktori_intelligence / "trusted_safe_domains_global.csv"

        if lokasi_trusted.exists():
            data = pd.read_csv(lokasi_trusted)
            if "domain" in data.columns:
                self.trusted_safe_domains = (
                    data["domain"]
                    .dropna()
                    .astype(str)
                    .str.lower()
                    .str.strip()
                    .unique()
                    .tolist()
                )

"""
    isi = wajib_replace(
        isi,
        "    def bersihkan_url(self, url: Any) -> str:\n",
        load_trusted + "    def bersihkan_url(self, url: Any) -> str:\n",
        "load trusted domains",
    )

if "def cek_trusted_safe_domain" not in isi:
    trusted_method = """
    def cek_trusted_safe_domain(self, domain: str) -> int:
        domain = str(domain).lower().strip()

        for trusted in getattr(self, "trusted_safe_domains", []):
            if domain == trusted or domain.endswith("." + trusted):
                return 1

        return 0

"""
    isi = wajib_replace(
        isi,
        "    def ekstrak_fitur_satu_url(self, url: Any) -> Dict[str, Any]:\n",
        trusted_method + "    def ekstrak_fitur_satu_url(self, url: Any) -> Dict[str, Any]:\n",
        "method cek trusted",
    )

patches = [
    (
        "        is_official = self.cek_domain_resmi(domain)\n",
        "        is_official = self.cek_domain_resmi(domain)\n        is_trusted_safe = self.cek_trusted_safe_domain(domain)\n",
        "is_trusted_safe = self.cek_trusted_safe_domain(domain)",
        "is_trusted_safe",
    ),
    (
        '            "is_official_domain": is_official,\n',
        '            "is_official_domain": is_official,\n            "trusted_safe_domain": is_trusted_safe,\n',
        '"trusted_safe_domain": is_trusted_safe,',
        "fitur trusted safe",
    ),
    (
        '        fitur_final["_lookalike_brand"] = lookalike_brand\n',
        '        fitur_final["_lookalike_brand"] = lookalike_brand\n        fitur_final["_trusted_safe_domain"] = is_trusted_safe\n',
        'fitur_final["_trusted_safe_domain"] = is_trusted_safe',
        "fitur final trusted",
    ),
    (
        '            "label_model_v5": "Berisiko" if probabilitas >= 0.5 else "Aman",\n',
        '            "label_model_v5": "Berisiko" if probabilitas >= 0.5 else "Aman",\n            "trusted_safe_domain": fitur.get("_trusted_safe_domain", 0),\n',
        '"trusted_safe_domain": fitur.get("_trusted_safe_domain", 0),',
        "output trusted",
    ),
    (
        '        is_official = int(float(hasil.get("is_official_domain", 0) or 0))\n',
        '        is_official = int(float(hasil.get("is_official_domain", 0) or 0))\n        is_trusted_safe = int(float(hasil.get("trusted_safe_domain", 0) or 0))\n',
        'is_trusted_safe = int(float(hasil.get("trusted_safe_domain", 0) or 0))',
        "kalibrasi trusted",
    ),
]

for lama, baru, penanda, nama in patches:
    if penanda not in isi:
        isi = wajib_replace(isi, lama, baru, nama)

if "Domain masuk daftar trusted safe" not in isi:
    lama = """        if brand_but_not_official:
            skor_final = max(skor_final, 82)
            alasan.append("Brand terdeteksi tetapi domain tidak cocok dengan daftar resmi.")
"""
    baru = """        if is_trusted_safe and suspicious_score == 0 and not lookalike_detected and not uses_punycode:
            public_ti_kuat = (
                "terindikasi" in public_ti_status
                or "ancaman" in public_ti_status
                or "malware" in public_ti_status
            )
            intelligence_kuat = (
                "tiruan_brand_berisiko" in intelligence_status
                or "domain_mirip_brand" in intelligence_status
                or "kata_mencurigakan_tinggi" in intelligence_status
            )

            if not public_ti_kuat and not intelligence_kuat:
                skor_final = min(skor_final, 24)
                brand_but_not_official = 0
                alasan.append("Domain masuk daftar trusted safe dan tidak memiliki sinyal phishing kuat.")

        if brand_but_not_official:
            skor_final = max(skor_final, 82)
            alasan.append("Brand terdeteksi tetapi domain tidak cocok dengan daftar resmi.")
"""
    isi = wajib_replace(isi, lama, baru, "blok brand false positive")

lokasi_engine_v5.write_text(isi, encoding="utf-8")

print("Patch false positive selesai.")
print("Engine:", lokasi_engine_v5)
print("Backup:", backup)


Patch false positive selesai.
Engine: C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py
Backup: C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5_backup_before_fp_patch_20260522_232429.py


In [4]:
if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import importlib
import phishrisk_engine_v5

importlib.reload(phishrisk_engine_v5)

engine_best = phishrisk_engine_v5.PhishRiskEngineV5(
    direktori_project=direktori_project,
    prefer_model="best",
)

print("Engine patch siap.")
print("Jumlah trusted domain:", len(getattr(engine_best, "trusted_safe_domains", [])))
print("Model:", engine_best.model_path)


Engine patch siap.
Jumlah trusted domain: 18
Model: C:\Users\ASUS\PHISHING\models\model_terbaik_multi_dataset_v5.pkl


In [5]:
data_uji_patch = pd.DataFrame([
    {"url": "https://pandas.pydata.org", "skenario": "trusted_netral", "expected": "Terlihat Aman"},
    {"url": "https://huggingface.co", "skenario": "trusted_netral", "expected": "Terlihat Aman"},
    {"url": "https://python.org", "skenario": "trusted_netral", "expected": "Terlihat Aman"},
    {"url": "https://scikit-learn.org", "skenario": "trusted_netral", "expected": "Terlihat Aman"},
    {"url": "https://kaggle.com", "skenario": "trusted_netral", "expected": "Terlihat Aman"},
    {"url": "https://wikipedia.org", "skenario": "trusted_netral", "expected": "Terlihat Aman"},
    {"url": "https://huggingface.co/login-update", "skenario": "trusted_dengan_sinyal", "expected": "Perlu Tinjauan"},
    {"url": "https://pandas.pydata.org/account-verify", "skenario": "trusted_dengan_sinyal", "expected": "Perlu Tinjauan"},
    {"url": "http://bca-login-update.test", "skenario": "berisiko", "expected": "Berisiko"},
    {"url": "http://micros0ft-login-update.test", "skenario": "berisiko", "expected": "Berisiko"},
    {"url": "https://xn--micrsoft-q4a.test", "skenario": "berisiko", "expected": "Berisiko"},
])

hasil_patch = []

for _, row in data_uji_patch.iterrows():
    hasil = engine_best.analisis_url(row["url"])
    hasil["skenario"] = row["skenario"]
    hasil["expected"] = row["expected"]
    hasil_patch.append(hasil)

data_hasil_patch = pd.DataFrame(hasil_patch)

def cek_status(row):
    if row["skenario"] == "trusted_netral":
        return "lolos" if row["hasil_akhir_v5"] == "Terlihat Aman" else "gagal"
    if row["skenario"] == "trusted_dengan_sinyal":
        return "lolos" if row["hasil_akhir_v5"] in ["Perlu Tinjauan", "Berisiko"] else "gagal"
    if row["skenario"] == "berisiko":
        return "lolos" if row["hasil_akhir_v5"] == "Berisiko" else "gagal"
    return "observasi"

data_hasil_patch["status_patch"] = data_hasil_patch.apply(cek_status, axis=1)

lokasi_hasil_patch = direktori_outputs / "hasil_patch_false_positive_engine_v5_step17b.csv"
data_hasil_patch.to_csv(lokasi_hasil_patch, index=False, encoding="utf-8")

kolom_tampil = [
    "url", "skenario", "expected", "trusted_safe_domain",
    "skor_model_v5", "skor_final_v5", "kategori_risiko_v5",
    "hasil_akhir_v5", "intelligence_status", "alasan_v5", "status_patch",
]
kolom_tampil = [kolom for kolom in kolom_tampil if kolom in data_hasil_patch.columns]

print("Uji patch selesai:", lokasi_hasil_patch)
display(data_hasil_patch[kolom_tampil])


Uji patch selesai: C:\Users\ASUS\PHISHING\reports\outputs\hasil_patch_false_positive_engine_v5_step17b.csv


,url,skenario,expected,trusted_safe_domain,skor_model_v5,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,intelligence_status,alasan_v5,status_patch
0,https://pandas.pydata.org,trusted_netral,Terlihat Aman,1,72.55,24.00,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
1,https://huggingface.co,trusted_netral,Terlihat Aman,1,4.18,82.00,Sangat Tinggi,Berisiko,brand_tidak_resmi_perlu_tinjauan,Brand terdeteksi tetapi domain tidak cocok den...,gagal
2,https://python.org,trusted_netral,Terlihat Aman,1,0.63,0.63,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
3,https://scikit-learn.org,trusted_netral,Terlihat Aman,1,2.37,2.37,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
4,https://kaggle.com,trusted_netral,Terlihat Aman,1,1.90,1.90,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
5,https://wikipedia.org,trusted_netral,Terlihat Aman,1,0.58,0.58,Rendah,Terlihat Aman,belum_ada_sinyal_kuat,Domain masuk daftar trusted safe dan tidak mem...,lolos
6,https://huggingface.co/login-update,trusted_dengan_sinyal,Perlu Tinjauan,1,36.27,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,URL memakai nama brand tetapi bukan domain res...,lolos
7,https://pandas.pydata.org/account-verify,trusted_dengan_sinyal,Perlu Tinjauan,1,7.30,72.00,Tinggi,Berisiko,kata_mencurigakan_tinggi,URL mengandung kata yang sering muncul pada se...,lolos
8,http://bca-login-update.test,berisiko,Berisiko,0,91.91,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,URL memakai nama brand tetapi bukan domain res...,lolos
9,http://micros0ft-login-update.test,berisiko,Berisiko,0,90.26,92.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,URL memakai nama brand tetapi bukan domain res...,lolos


In [6]:
data_step17 = pd.read_csv(lokasi_step17)
data_validasi_ulang = []

for _, row in data_step17[["url", "skenario", "expected"]].iterrows():
    hasil = engine_best.analisis_url(row["url"])
    hasil["skenario"] = row["skenario"]
    hasil["expected"] = row["expected"]
    data_validasi_ulang.append(hasil)

data_validasi_ulang = pd.DataFrame(data_validasi_ulang)

data_before_after = data_step17[[
    "url", "skenario", "expected", "skor_final_v5", "kategori_risiko_v5", "hasil_akhir_v5"
]].rename(columns={
    "skor_final_v5": "skor_final_before",
    "kategori_risiko_v5": "kategori_before",
    "hasil_akhir_v5": "hasil_before",
}).merge(
    data_validasi_ulang[[
        "url", "trusted_safe_domain", "skor_final_v5", "kategori_risiko_v5", "hasil_akhir_v5", "alasan_v5"
    ]].rename(columns={
        "skor_final_v5": "skor_final_after",
        "kategori_risiko_v5": "kategori_after",
        "hasil_akhir_v5": "hasil_after",
        "alasan_v5": "alasan_after",
    }),
    on="url",
    how="left",
)

data_before_after["hasil_berubah"] = data_before_after["hasil_before"] != data_before_after["hasil_after"]

def status_after(row):
    if row["skenario"] == "resmi":
        return "lolos" if row["hasil_after"] in ["Terlihat Aman", "Perlu Tinjauan"] else "gagal"
    if row["skenario"] == "berisiko":
        return "lolos" if row["hasil_after"] == "Berisiko" else "gagal"
    return "observasi"

data_before_after["status_validasi_after"] = data_before_after.apply(status_after, axis=1)

lokasi_before_after = direktori_outputs / "perbandingan_before_after_patch_fp_engine_v5_step17b.csv"
lokasi_ringkasan_patch = direktori_outputs / "ringkasan_patch_false_positive_engine_v5_step17b.csv"
lokasi_gagal_after = direktori_outputs / "temuan_gagal_after_patch_engine_v5_step17b.csv"

data_before_after.to_csv(lokasi_before_after, index=False, encoding="utf-8")

ringkasan_patch = (
    data_before_after
    .groupby(["skenario", "status_validasi_after"])
    .size()
    .reset_index(name="jumlah_data")
)

temuan_gagal_after = data_before_after[data_before_after["status_validasi_after"] == "gagal"].copy()

ringkasan_patch.to_csv(lokasi_ringkasan_patch, index=False, encoding="utf-8")
temuan_gagal_after.to_csv(lokasi_gagal_after, index=False, encoding="utf-8")

print("Perbandingan before-after selesai:", lokasi_before_after)
display(data_before_after[data_before_after["hasil_berubah"]])
display(ringkasan_patch)
display(temuan_gagal_after)


Perbandingan before-after selesai: C:\Users\ASUS\PHISHING\reports\outputs\perbandingan_before_after_patch_fp_engine_v5_step17b.csv


,url,skenario,expected,skor_final_before,kategori_before,hasil_before,trusted_safe_domain,skor_final_after,kategori_after,hasil_after,alasan_after,hasil_berubah,status_validasi_after
34,https://pandas.pydata.org,netral,Tidak Dipaksa,72.55,Tinggi,Berisiko,1,24.0,Rendah,Terlihat Aman,Domain masuk daftar trusted safe dan tidak mem...,True,observasi


,skenario,status_validasi_after,jumlah_data
0,berisiko,lolos,15
1,netral,observasi,8
2,resmi,lolos,15


,url,skenario,expected,skor_final_before,kategori_before,hasil_before,trusted_safe_domain,skor_final_after,kategori_after,hasil_after,alasan_after,hasil_berubah,status_validasi_after


In [7]:
input_cli = direktori_examples / "input_url_step17b_patch_engine_v5.csv"
output_cli = direktori_outputs / "hasil_cli_step17b_patch_engine_v5.csv"

data_uji_patch[["url"]].to_csv(input_cli, index=False, encoding="utf-8")

perintah = [
    sys.executable,
    str(lokasi_cli_v5),
    "--mode", "urls",
    "--input", str(input_cli),
    "--url-column", "url",
    "--output", str(output_cli),
    "--model-mode", "best",
]

hasil_cli = subprocess.run(perintah, capture_output=True, text=True)

print("Return code:", hasil_cli.returncode)
print("STDOUT:")
print(hasil_cli.stdout)
print("STDERR:")
print(hasil_cli.stderr)

if hasil_cli.returncode != 0:
    raise RuntimeError("CLI V5 gagal setelah patch.")

data_cli = pd.read_csv(output_cli)
kolom_cli = [
    "url", "trusted_safe_domain", "skor_final_v5",
    "kategori_risiko_v5", "hasil_akhir_v5", "engine_version"
]
kolom_cli = [kolom for kolom in kolom_cli if kolom in data_cli.columns]

display(data_cli[kolom_cli])


Return code: 0
STDOUT:
PhishRisk Engine V5 selesai.
Mode: urls
Output: C:\Users\ASUS\PHISHING\reports\outputs\hasil_cli_step17b_patch_engine_v5.csv
hasil_akhir_v5 kategori_risiko_v5  skor_final_v5 engine_version
 Terlihat Aman             Rendah          24.00             V5
      Berisiko      Sangat Tinggi          82.00             V5
 Terlihat Aman             Rendah           0.63             V5
 Terlihat Aman             Rendah           2.37             V5
 Terlihat Aman             Rendah           1.90             V5
 Terlihat Aman             Rendah           0.58             V5
      Berisiko      Sangat Tinggi          92.00             V5
      Berisiko             Tinggi          72.00             V5
      Berisiko      Sangat Tinggi          92.00             V5
      Berisiko      Sangat Tinggi          92.00             V5

STDERR:



,url,trusted_safe_domain,skor_final_v5,kategori_risiko_v5,hasil_akhir_v5,engine_version
0,https://pandas.pydata.org,1,24.00,Rendah,Terlihat Aman,V5
1,https://huggingface.co,1,82.00,Sangat Tinggi,Berisiko,V5
2,https://python.org,1,0.63,Rendah,Terlihat Aman,V5
3,https://scikit-learn.org,1,2.37,Rendah,Terlihat Aman,V5
4,https://kaggle.com,1,1.90,Rendah,Terlihat Aman,V5
5,https://wikipedia.org,1,0.58,Rendah,Terlihat Aman,V5
6,https://huggingface.co/login-update,1,92.00,Sangat Tinggi,Berisiko,V5
7,https://pandas.pydata.org/account-verify,1,72.00,Tinggi,Berisiko,V5
8,http://bca-login-update.test,0,92.00,Sangat Tinggi,Berisiko,V5
9,http://micros0ft-login-update.test,0,92.00,Sangat Tinggi,Berisiko,V5


In [8]:
jumlah_patch_gagal = int((data_hasil_patch["status_patch"] == "gagal").sum())
jumlah_gagal_after = int(len(temuan_gagal_after))

status_patch_final = (
    "patch_siap"
    if jumlah_patch_gagal == 0 and jumlah_gagal_after == 0 and hasil_cli.returncode == 0
    else "perlu_tinjauan"
)

data_status_patch = pd.DataFrame([{
    "status_patch_final": status_patch_final,
    "jumlah_uji_patch_gagal": jumlah_patch_gagal,
    "jumlah_validasi_step17_gagal_after": jumlah_gagal_after,
    "cli_return_code": int(hasil_cli.returncode),
    "trusted_domain_count": len(getattr(engine_best, "trusted_safe_domains", [])),
    "catatan": (
        "Patch siap. Engine V5 dapat lanjut ke integrasi Streamlit."
        if status_patch_final == "patch_siap"
        else "Patch masih perlu ditinjau."
    ),
}])

lokasi_status_patch = direktori_outputs / "status_patch_false_positive_engine_v5_step17b.csv"
data_status_patch.to_csv(lokasi_status_patch, index=False, encoding="utf-8")

metadata = {
    "nama_notebook": "17B_patch_false_positive_engine_v5.ipynb",
    "nama_tahap": "Patch False Positive Engine V5",
    "status": status_patch_final,
    "trusted_domains_file": str(lokasi_trusted),
    "engine_patch_file": str(lokasi_engine_v5),
    "engine_backup_file": str(backup),
    "hasil_patch_test": str(lokasi_hasil_patch),
    "perbandingan_before_after": str(lokasi_before_after),
    "status_patch": str(lokasi_status_patch),
    "catatan": [
        "Patch menurunkan risiko trusted safe domain hanya jika tidak ada sinyal phishing kuat.",
        "Patch tidak melakukan retraining model.",
        "Patch ditujukan untuk false positive netral seperti pandas.pydata.org dan huggingface.co.",
    ],
    "tanggal_selesai": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

lokasi_metadata = direktori_outputs / "metadata_step17b_patch_false_positive_engine_v5.json"
lokasi_metadata.write_text(json.dumps(metadata, indent=4, ensure_ascii=False), encoding="utf-8")

catatan = f"""
CATATAN FINAL STEP 17B

Notebook:
17B_patch_false_positive_engine_v5.ipynb

Status:
{status_patch_final}

Output utama:
{lokasi_status_patch}

Catatan:
Patch false positive Engine V5 selesai.
Patch bukan whitelist buta. URL dengan sinyal kuat tetap dapat naik risiko.

Tahap berikutnya:
18_integrasi_streamlit_engine_v5.ipynb
"""

lokasi_catatan = direktori_outputs / "catatan_final_step17b_patch_false_positive_engine_v5.txt"
lokasi_catatan.write_text(catatan, encoding="utf-8")

print(catatan)
print("Metadata:", lokasi_metadata)
display(data_status_patch)



CATATAN FINAL STEP 17B

Notebook:
17B_patch_false_positive_engine_v5.ipynb

Status:
perlu_tinjauan

Output utama:
C:\Users\ASUS\PHISHING\reports\outputs\status_patch_false_positive_engine_v5_step17b.csv

Catatan:
Patch false positive Engine V5 selesai.
Patch bukan whitelist buta. URL dengan sinyal kuat tetap dapat naik risiko.

Tahap berikutnya:
18_integrasi_streamlit_engine_v5.ipynb

Metadata: C:\Users\ASUS\PHISHING\reports\outputs\metadata_step17b_patch_false_positive_engine_v5.json


,status_patch_final,jumlah_uji_patch_gagal,jumlah_validasi_step17_gagal_after,cli_return_code,trusted_domain_count,catatan
0,perlu_tinjauan,1,0,0,18,Patch masih perlu ditinjau.


In [9]:
file_output = [
    lokasi_trusted,
    lokasi_engine_v5,
    backup,
    lokasi_hasil_patch,
    lokasi_before_after,
    lokasi_ringkasan_patch,
    lokasi_gagal_after,
    input_cli,
    output_cli,
    lokasi_status_patch,
    lokasi_metadata,
    lokasi_catatan,
]

validasi_output = pd.DataFrame([{
    "nama_file": Path(file).name,
    "lokasi": str(file),
    "tersedia": Path(file).exists(),
    "ukuran_kb": round(Path(file).stat().st_size / 1024, 2) if Path(file).exists() else 0,
} for file in file_output])

lokasi_validasi = direktori_outputs / "validasi_step17b_patch_false_positive_engine_v5.csv"
validasi_output.to_csv(lokasi_validasi, index=False, encoding="utf-8")

print("Validasi output STEP 17B:", lokasi_validasi)
display(validasi_output)


Validasi output STEP 17B: C:\Users\ASUS\PHISHING\reports\outputs\validasi_step17b_patch_false_positive_engine_v5.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,trusted_safe_domains_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\trust...,True,1.00
1,phishrisk_engine_v5.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5.py,True,26.65
2,phishrisk_engine_v5_backup_before_fp_patch_202...,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v5...,True,24.69
3,hasil_patch_false_positive_engine_v5_step17b.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_p...,True,18.14
4,perbandingan_before_after_patch_fp_engine_v5_s...,C:\Users\ASUS\PHISHING\reports\outputs\perband...,True,8.27
5,ringkasan_patch_false_positive_engine_v5_step1...,C:\Users\ASUS\PHISHING\reports\outputs\ringkas...,True,0.10
6,temuan_gagal_after_patch_engine_v5_step17b.csv,C:\Users\ASUS\PHISHING\reports\outputs\temuan_...,True,0.18
7,input_url_step17b_patch_engine_v5.csv,C:\Users\ASUS\PHISHING\examples\input_url_step...,True,0.31
8,hasil_cli_step17b_patch_engine_v5.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_c...,True,17.75
9,status_patch_false_positive_engine_v5_step17b.csv,C:\Users\ASUS\PHISHING\reports\outputs\status_...,True,0.17
